In [ ]:
%pip install fastparquet

# Constrained sOED Planning
This notebook is isolated from exploratory code and uses a dedicated constrained agent.

Hard constraints enforced during planning:
- Mass1 < 30
- Mass1 + Mass2 < 30
- Mass1 + Mass2 >= 3 (minimum load wall)
- Boost pressure within ±0.5 bar of: Boost = 0.0922 * (Mass1 + Mass2) + 0.8378
- Boost pressure >= 1.0 bar (ambient floor)
- Boost pressure <= 3.8 bar (TC roof)
- BR limit by load band:
  - 0 < Mass1 < 10: 0.5 < Mass2 < 3.5
  - 10 < Mass1 < 20: 0.9 < Mass2 < 3.0
  - 20 < Mass1 < 30: 0.0 < Mass2 < 1.5
- VVA limit by load band:
  - 0 < Mass1 < 10: IVO 350-435, IVC 500-540, EVO 128-218, EVC 270-350
  - 10 <= Mass1 < 20: IVO 330-390, IVC 500-570, EVO 128-218, EVC 330-370
  - 20 <= Mass1 < 30: IVO 345-365, IVC 495-535, EVO 128-218, EVC 345-355

The notebook keeps these as hard feasibility checks on candidate paths.

In [ ]:
%cd D:/shahnawaz/uva/main
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.data_loader import DataLoader
from src.scaler import Scaler
from src.constant_manager import ConstantManager
from src.soed.agents.constrained_multistep_mimo_agent import ConstrainedMultiStepMIMOAgent

import importlib

import src.soed.agents.constrained_multistep_mimo_agent_fixed as fixed_mod
importlib.reload(fixed_mod)

from src.soed.agents.constrained_multistep_mimo_agent_fixed import ConstrainedMultiStepMIMOAgentFixed

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

In [ ]:
PD_DATA_FILE = 'rcci_cleaned_data_v4_5.parquet'

cm = ConstantManager()
input_features = cm.RAW_REDUCED_INPUT_COLUMNS
output_features = cm.RAW_OUTPUT_COLUMNS

required = {'Boost pressure', 'Mass1', 'Mass2'}
missing = required.difference(set(input_features))
if missing:
    raise ValueError(f'Missing required constrained features in input_features: {missing}')

pd_df = DataLoader(file_path=PD_DATA_FILE).load_data()

scaler_x = Scaler()
scaler_y = Scaler()

scaled_df = pd_df[input_features + output_features].copy()
scaled_df[input_features] = scaler_x.fit_transform(scaled_df, input_features)
scaled_df[output_features] = scaler_y.fit_transform(scaled_df, output_features)

inputs_scaled = scaled_df[input_features].to_numpy()
outputs_scaled = scaled_df[output_features].to_numpy()

scaled_df[input_features + output_features].head()

In [ ]:
# Build bounds from unscaled engineering bounds, then map to scaled domain
unscaled_bounds = cm.UNSCALED_BOUNDS
ordered_bounds = [unscaled_bounds[f] for f in input_features]

# Keep feature names attached so StandardScaler does not warn about missing names.
unscaled_bounds_df = pd.DataFrame(
    np.array(ordered_bounds).T,
    columns=input_features,
)
scaled_bounds_df = pd.DataFrame(
    scaler_x.transform(unscaled_bounds_df),
    columns=input_features,
)
scaled_bounds = torch.tensor(scaled_bounds_df.to_numpy().T, dtype=torch.float64)

print("scaled_bounds shape:", tuple(scaled_bounds.shape))
scaled_bounds

In [ ]:
# Quick data sanity check before GP fit
x_arr = np.asarray(inputs_scaled, dtype=float)
y_arr = np.asarray(outputs_scaled, dtype=float)

x_bad = ~np.isfinite(x_arr)
y_bad = ~np.isfinite(y_arr)

print("Inputs shape:", x_arr.shape, "| Outputs shape:", y_arr.shape)
print("Input non-finite count:", int(x_bad.sum()))
print("Output non-finite count:", int(y_bad.sum()))

if x_bad.any():
    bad_cols_x = [input_features[i] for i in np.where(x_bad.any(axis=0))[0]]
    print("Input columns with NaN/Inf:", bad_cols_x)

if y_bad.any():
    bad_cols_y = [output_features[i] for i in np.where(y_bad.any(axis=0))[0]]
    print("Output columns with NaN/Inf:", bad_cols_y)

print("Rows fully finite:", int((np.isfinite(x_arr).all(axis=1) & np.isfinite(y_arr).all(axis=1)).sum()))

In [ ]:
from pathlib import Path
import gpytorch

MODEL_BUNDLE_PATH = Path("model_artifacts/constrained_agent_bundle_v1.pth")
REUSE_SAVED_MODELS = True

# Planner behavior knobs
USE_DISTANCE_PENALTY = True
DISTANCE_WEIGHT = 0.05  # used only when USE_DISTANCE_PENALTY=True
planner_w_dist = DISTANCE_WEIGHT if USE_DISTANCE_PENALTY else 0.0

# Additional freedom knobs
INCLUDE_LOCAL_PATHS = True
LOCAL_PATH_FRACTION = 0.10  # local candidates as fraction of Sobol count
FEASIBLE_MARGIN_WEIGHT = 0.0

# Feasibility retry knobs
REQUIRE_FULLY_FEASIBLE_PATH = True
MAX_PLAN_RETRIES = 3
BASE_NUM_SCENARIOS = 2048
SCENARIO_GROWTH_PER_RETRY = 0.30  # grows candidate count each retry

agent = ConstrainedMultiStepMIMOAgent(
    bounds=scaled_bounds,
    feature_names=input_features,
    scaler_x=scaler_x,
    mass1_name='Mass1',
    mass2_name='Mass2',
    boost_name='Boost pressure',
    load_limit=30.0,
    boost_slope=0.0922,
    boost_intercept=0.8378,
    boost_band=0.5,
    min_load=3.0,
    ambient_pressure=1.0,
    tc_boost_limit=3.8,
 )

if REUSE_SAVED_MODELS and MODEL_BUNDLE_PATH.exists():
    loaded = agent.load_bundle(MODEL_BUNDLE_PATH)
    print(f"Loaded constrained surrogate bundle from: {MODEL_BUNDLE_PATH}")
    print(f"Bundle metadata keys: {list(loaded.keys())}")
else:
    agent.fit_data(inputs_scaled, outputs_scaled)
    saved_path = agent.save_bundle(
        MODEL_BUNDLE_PATH,
        extra_metadata={
            "source_notebook": "soed_constrained_planning.ipynb",
            "input_features": input_features,
            "output_features": output_features,
        },
    )
    print(f"Trained and saved constrained surrogate bundle to: {saved_path}")

print(f"Distance penalty enabled: {USE_DISTANCE_PENALTY} | w_dist={planner_w_dist}")
print(f"Include local RW paths: {INCLUDE_LOCAL_PATHS} | local_path_fraction={LOCAL_PATH_FRACTION}")
print(f"Feasible interior margin weight: {FEASIBLE_MARGIN_WEIGHT}")
print(f"Require fully feasible path: {REQUIRE_FULLY_FEASIBLE_PATH} | max retries: {MAX_PLAN_RETRIES}")

def _path_is_hard_feasible(a, p):
    with torch.no_grad():
        feasible_mask = a._feasible_mask(p.unsqueeze(0))
    return bool(feasible_mask.reshape(-1)[0].item())

def _path_objective(a, p, curr, w_dist, feasible_margin_weight):
    p_batch = p.unsqueeze(0)
    with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-4), gpytorch.settings.fast_pred_var(False):
        total_ig = torch.zeros(1, dtype=torch.float64, device=p.device)
        for model, lik in zip(a.models, a.likelihoods):
            cov = model.posterior(p_batch).distribution.covariance_matrix
            if cov.ndim == 4:
                cov = cov.squeeze(0)
            if not torch.isfinite(cov).all():
                return -float('inf')
            m = torch.eye(p.shape[0], dtype=cov.dtype, device=cov.device) + cov / lik.noise.clamp_min(1e-8)
            try:
                ig = torch.linalg.cholesky(m).diagonal(dim1=-2, dim2=-1).log().sum(dim=-1)
            except RuntimeError:
                ig = 0.5 * torch.linalg.slogdet(m)[1]
            total_ig += torch.where(torch.isfinite(ig), ig, torch.full_like(ig, -1e6))

        ls_eff = torch.min(
            torch.stack([m.covar_module.base_kernel.lengthscale.squeeze().detach() for m in a.models]),
            dim=0,
        ).values
        d0 = torch.sqrt((((p[0] - curr) / ls_eff) ** 2).sum())
        dstep = torch.sqrt((((p[1:] - p[:-1]) / ls_eff) ** 2).sum(dim=-1)).sum() if p.shape[0] > 1 else 0.0
        score = total_ig.squeeze() - w_dist * (d0 + dstep)

        feasible = _path_is_hard_feasible(a, p)
        if feasible:
            score = score - float(feasible_margin_weight) * a._interior_margin_penalty(p_batch).squeeze()
        else:
            score = score - 50.0 * a._constraint_penalty(p_batch).squeeze() - 10.0 * a._interior_margin_penalty(p_batch).squeeze()
        return float(score.item())

current_loc = agent.X[-1:]
curr = current_loc.squeeze()
attempt_records = []

for attempt in range(1, MAX_PLAN_RETRIES + 1):
    num_scenarios = int(round(BASE_NUM_SCENARIOS * (1.0 + SCENARIO_GROWTH_PER_RETRY * (attempt - 1))))
    candidate = agent.plan_multistep_batch(
        current_location=current_loc,
        q_steps=3,
        num_scenarios=num_scenarios,
        w_dist=planner_w_dist,
        enforce_feasible_sampling=True,
        enforce_feasible_sobol=True,
        include_local_paths=INCLUDE_LOCAL_PATHS,
        local_path_fraction=LOCAL_PATH_FRACTION,
        feasible_margin_weight=FEASIBLE_MARGIN_WEIGHT,
    )

    is_feasible = _path_is_hard_feasible(agent, candidate)
    score = _path_objective(
        agent,
        candidate,
        curr=curr,
        w_dist=planner_w_dist,
        feasible_margin_weight=FEASIBLE_MARGIN_WEIGHT,
    )

    attempt_records.append({
        'attempt': attempt,
        'num_scenarios': num_scenarios,
        'path': candidate.detach().clone(),
        'hard_feasible': is_feasible,
        'objective_score': score,
    })
    print(f"Attempt {attempt:02d}: scenarios={num_scenarios} | hard-feasible={is_feasible} | score={score:.4f}")

if REQUIRE_FULLY_FEASIBLE_PATH:
    feasible_attempts = [r for r in attempt_records if r['hard_feasible']]
    if feasible_attempts:
        best = max(feasible_attempts, key=lambda r: r['objective_score'])
        path_scaled = best['path']
        print(
            f"Selected best feasible attempt {best['attempt']} "
            f"(scenarios={best['num_scenarios']}, score={best['objective_score']:.4f})"
        )
    else:
        best = max(attempt_records, key=lambda r: r['objective_score'])
        path_scaled = best['path']
        print("Warning: retries exhausted; returning best available soft-constrained path.")
        print(
            f"Selected fallback attempt {best['attempt']} "
            f"(scenarios={best['num_scenarios']}, score={best['objective_score']:.4f})"
        )
else:
    best = max(attempt_records, key=lambda r: r['objective_score'])
    path_scaled = best['path']
    print(
        f"Selected best overall attempt {best['attempt']} "
        f"(scenarios={best['num_scenarios']}, score={best['objective_score']:.4f}, "
        f"hard-feasible={best['hard_feasible']})"
    )

path_feasible = _path_is_hard_feasible(agent, path_scaled)
last_attempt = int(best['attempt'])

path_scaled

In [ ]:
# Convert recommendation to physical units and verify all constraints
path_original = []
for p in path_scaled:
    row = []
    for i in range(len(input_features)):
        row.append(scaler_x.inverse_transform(p[i].item(), i))
    path_original.append(row)

path_original = pd.DataFrame(path_original, columns=input_features)

mass_sum = path_original['Mass1'] + path_original['Mass2']
boost_center = 0.0922 * mass_sum + 0.8378

# Pull hard bounds from agent so validation mirrors planner logic.
min_load = float(getattr(agent, 'min_load', 3.0))
max_load = float(getattr(agent, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent, 'tc_boost_limit', 3.8))

path_original['Mass1+Mass2'] = mass_sum
path_original['Boost_low_raw'] = boost_center - 0.5
path_original['Boost_high_raw'] = boost_center + 0.5
path_original['Boost_low'] = np.maximum(path_original['Boost_low_raw'], ambient_pressure)
path_original['Boost_high'] = np.minimum(path_original['Boost_high_raw'], tc_boost_limit)

# Load check aligned with planner constraints.
path_original['Load_ok'] = (
    (path_original['Mass1'] >= 0.0) &
    (path_original['Mass1'] < max_load) &
    (mass_sum >= min_load) &
    (mass_sum < max_load)
)

# Boost check uses slanted band clipped by floor/roof, aligned with planner.
path_original['Boost_ok'] = (
    (path_original['Boost pressure'] >= path_original['Boost_low']) &
    (path_original['Boost pressure'] <= path_original['Boost_high'])
)

# BR limit by Mass1 band (aligned with planner band edges).
path_original['BR_ok'] = (
    ((path_original['Mass1'] >= 0.0) & (path_original['Mass1'] < 10.0) & (path_original['Mass2'] > 0.5) & (path_original['Mass2'] < 3.5)) |
    ((path_original['Mass1'] >= 10.0) & (path_original['Mass1'] < 20.0) & (path_original['Mass2'] > 0.9) & (path_original['Mass2'] < 3.0)) |
    ((path_original['Mass1'] >= 20.0) & (path_original['Mass1'] < 30.0) & (path_original['Mass2'] > 0.0) & (path_original['Mass2'] < 1.5))
)

# VVA limit by Mass1 band
low_load = (path_original['Mass1'] >= 0.0) & (path_original['Mass1'] < 10.0)
mid_load = (path_original['Mass1'] >= 10.0) & (path_original['Mass1'] < 20.0)
high_load = (path_original['Mass1'] >= 20.0) & (path_original['Mass1'] < 30.0)

path_original['VVA_ok'] = (
    (
        low_load &
        (path_original['IVO'] >= 350.0) & (path_original['IVO'] <= 435.0) &
        (path_original['IVC'] >= 500.0) & (path_original['IVC'] <= 540.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 270.0) & (path_original['EVC'] <= 350.0)
    ) |
    (
        mid_load &
        (path_original['IVO'] >= 330.0) & (path_original['IVO'] <= 390.0) &
        (path_original['IVC'] >= 500.0) & (path_original['IVC'] <= 570.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 330.0) & (path_original['EVC'] <= 370.0)
    ) |
    (
        high_load &
        (path_original['IVO'] >= 345.0) & (path_original['IVO'] <= 365.0) &
        (path_original['IVC'] >= 495.0) & (path_original['IVC'] <= 535.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 345.0) & (path_original['EVC'] <= 355.0)
    )
)

path_original['Feasible'] = path_original['Load_ok'] & path_original['Boost_ok'] & path_original['BR_ok'] & path_original['VVA_ok']

# Compact display columns for decision making.
display_cols = [
    'Engine_speed', 'Boost pressure', 'Mass1', 'Mass2', 'SOI2', 'IVO', 'IVC', 'EVO', 'EVC',
    'Mass1+Mass2', 'Boost_low', 'Boost_high', 'Load_ok', 'Boost_ok', 'BR_ok', 'VVA_ok', 'Feasible'
]
path_original[display_cols]

In [ ]:
# 2D engineering constraint view: Boost vs (Mass1+Mass2) with explicit polygon boundaries
hist = pd_df.copy()
hist['Mass1+Mass2'] = hist['Mass1'] + hist['Mass2']

x = np.linspace(hist['Mass1+Mass2'].min(), max(40.0, hist['Mass1+Mass2'].max()), 400)
y_mid = 0.0922 * x + 0.8378
y_low = y_mid - 0.5
y_high = y_mid + 0.5

# Pull polygon caps from the current agent if available
min_load = float(getattr(agent, 'min_load', 3.0))
max_load = float(getattr(agent, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent, 'tc_boost_limit', 3.8))

# Polygon is the intersection of: boost band, floor/roof, and left/right load walls
poly_low = np.maximum(y_low, ambient_pressure)
poly_high = np.minimum(y_high, tc_boost_limit)
poly_mask = (x >= min_load) & (x <= max_load) & (poly_high >= poly_low)

plt.figure(figsize=(10, 6))
plt.scatter(hist['Mass1+Mass2'], hist['Boost pressure'], s=25, c='#3b6fb6', alpha=0.65, label='Historical cases')

# Slanted boost corridor lines
plt.plot(x, y_mid, '--', c='gray', lw=1.5, label='Boost center line')
plt.plot(x, y_low, '--', c='red', lw=1.5, label='Boost lower slanted limit')
plt.plot(x, y_high, '--', c='red', lw=1.5, label='Boost upper slanted limit')

# Horizontal roof/floor
plt.axhline(ambient_pressure, color='purple', linestyle='-.', lw=1.4, label='Ambient pressure floor')
plt.axhline(tc_boost_limit, color='brown', linestyle='-.', lw=1.4, label='TC boost roof')

# Vertical load walls
plt.axvline(min_load, color='black', linestyle='-', lw=1.4, label='Min load wall')
plt.axvline(max_load, color='black', linestyle='-', lw=1.4, label='Max load wall')

# Filled feasible polygon region
plt.fill_between(
    x[poly_mask],
    poly_low[poly_mask],
    poly_high[poly_mask],
    color='limegreen',
    alpha=0.18,
    label='Feasible polygon region',
)

rec_x = path_original['Mass1+Mass2'].to_numpy()
rec_y = path_original['Boost pressure'].to_numpy()
plt.plot(rec_x, rec_y, '-o', c='orangered', lw=2.6, label='Recommended path')

for i, (xx, yy) in enumerate(zip(rec_x, rec_y), start=1):
    plt.text(xx + 0.25, yy + 0.02, f'Step {i}', color='orangered')

plt.xlabel('Mass1 + Mass2 [kg/h]')
plt.ylabel('Boost pressure [bar]')
plt.title('Constrained Planning Polygon and Recommended Steps')
plt.grid(alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# --- setup ---
import torch
import numpy as np
import pandas as pd

from src.soed.agents.constrained_multistep_mimo_agent import ConstrainedMultiStepMIMOAgent
from src.soed.agents.constrained_multistep_mimo_agent_fixed import ConstrainedMultiStepMIMOAgentFixed
from src.soed.agents.full_fixed_planner_pipeline import run_fixed_planner_pipeline

# make sure notebook variables exist
if "current_loc" not in globals():
    current_loc = torch.tensor([12.0, 2.3, 1.5, 360.0, 520.0, 150.0, 320.0], dtype=torch.float64)

if "input_features" not in globals():
    input_features = [
        "Mass1",
        "Mass2",
        "Boost pressure",
        "IVO",
        "IVC",
        "EVO",
        "EVC",
    ]

if "output_features" not in globals():
    output_features = [
        "CH4",
        "NMHC",
        "CO",
        "Pmax",
        "PRR4_max",
        "CA50",
        "IEMP",
        "ITE",
        "Lambda",
        "CO2",
        "Nox",
    ]

if "scaled_df" not in globals():
    raise ValueError("df is not defined. Run the dataset prep cells first.")

if "inputs_scaled" not in globals():
    inputs_scaled = torch.tensor(df[input_features].to_numpy(dtype=float), dtype=torch.float64)

if "outputs_scaled" not in globals():
    outputs_scaled = torch.tensor(df[output_features].to_numpy(dtype=float), dtype=torch.float64)

if "scaled_bounds" not in globals():
    # use same bound construction pattern as existing notebook
    ordered_bounds = []
    for feat in input_features:
        ordered_bounds.append([float(df[feat].min()), float(df[feat].max())])
    scaled_bounds = torch.tensor(np.array(ordered_bounds).T, dtype=torch.float64)

print("Using input_features:", input_features)
print("Using output_features:", output_features)
print("Current location:", current_loc)

In [ ]:
# --- Experiment 1: current planner as-is ---
agent_old = ConstrainedMultiStepMIMOAgent(
    bounds=scaled_bounds,
    feature_names=input_features,
    scaler_x=None,
    mass1_name="Mass1",
    mass2_name="Mass2",
    boost_name="Boost pressure",
    load_limit=30.0,
    boost_slope=0.0922,
    boost_intercept=0.8378,
    boost_band=0.5,
    min_load=3.0,
    ambient_pressure=1.0,
    tc_boost_limit=3.8,
)

agent_old.fit_data(inputs_scaled, outputs_scaled)
path_old = agent_old.plan_multistep_batch(
    current_location=current_loc,
    q_steps=3,
    num_scenarios=256,
    w_dist=0.05,
    enforce_feasible_sampling=True,
    enforce_feasible_sobol=True,
    include_local_paths=True,
    local_path_fraction=0.10,
    feasible_margin_weight=0.0,
)

print("Experiment 1: current planner path")
print(path_old)

In [ ]:
# --- Experiment 2: fixed planner prototype ---
agent_fixed = ConstrainedMultiStepMIMOAgentFixed(
    bounds=scaled_bounds,
    feature_names=input_features,
    scaler_x=None,
    mass1_name="Mass1",
    mass2_name="Mass2",
    boost_name="Boost pressure",
    load_limit=30.0,
    boost_slope=0.0922,
    boost_intercept=0.8378,
    boost_band=0.5,
    min_load=3.0,
    ambient_pressure=1.0,
    tc_boost_limit=3.8,
)

agent_fixed.fit_data(inputs_scaled, outputs_scaled)
path_fixed = agent_fixed.plan_multistep_batch(
    current_location=current_loc,
    q_steps=3,
    num_scenarios=256,
    w_dist=1.0,
    enforce_feasible_sampling=False,
    feasible_margin_weight=25.0,
)

In [ ]:
# --- Experiment 3: full fixed pipeline ---
result = run_fixed_planner_pipeline(
    df=scaled_df,
    input_features=input_features,
    output_features=output_features,
    current_location=current_loc,
    q_steps=3,
    num_scenarios=256,
)

print("Experiment 3: full fixed pipeline path")
print(result["path_original_units"])

In [ ]:
# --- compact comparison summary ---
print("Old feasible:", bool(agent_old._feasible_mask(path_old.unsqueeze(0)).item()))
print("Fixed feasible:", bool(agent_fixed._feasible_mask(path_fixed.unsqueeze(0)).item()))
print("Old path shape:", tuple(path_old.shape))
print("Fixed path shape:", tuple(path_fixed.shape))
print("Pipeline path shape:", tuple(result["path_original_units"].shape))